# F1 Moodle Activity: Predicting Students at Risk

## Modeling

This notebook develops models for identifying students who are at risk of failing the course based on their Moodle activity.

At this stage, the main feature is `active_days`, which measures the number of days a student was active in Moodle before one week before the midterm.

The models are trained on students from 2019 and 2021–2024 and tested on students from 2025–2026.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score

## 1. Data preparation

The data used at this stage comes from HW7. It contains students' course results and their Moodle activity before the midterm.

For this first modeling stage, we use `active_days` as the main predictor.

In [5]:
data = pd.read_csv("../data/active_days_hw7.csv")

print(data.shape)
print(data.columns)
display(data.head())

(843, 44)
Index(['course_run_id', 'actor_id', 'homework_01_eng', 'homework_01_est',
       'homework_02_eng', 'homework_02_est', 'homework_03_eng',
       'homework_03_est', 'homework_04_eng', 'homework_04_est',
       'homework_05_eng', 'homework_05_est', 'homework_06_eng',
       'homework_06_est', 'homework_07_eng', 'homework_07_est',
       'homework_08_eng', 'homework_08_est', 'homework_09_eng',
       'homework_09_est', 'homework_10_eng', 'homework_10_est',
       'homework_11_eng', 'homework_11_est', 'homework_12_eng',
       'homework_12_est', 'midterm_attempt_1_eng', 'midterm_attempt_1_est',
       'midterm_attempt_2_eng', 'midterm_attempt_2_est',
       'midterm_attempt_3_eng', 'midterm_attempt_3_est',
       'final_exam_attempt_1_eng', 'final_exam_attempt_1_est',
       'final_exam_attempt_2_eng', 'final_exam_attempt_2_est',
       'final_exam_attempt_3_eng', 'final_exam_attempt_3_est', 'bonus',
       'course_numeric_grade', 'course_letter_grade', 'year', 'active_days',
   

,course_run_id,actor_id,homework_01_eng,homework_01_est,homework_02_eng,homework_02_est,homework_03_eng,homework_03_est,homework_04_eng,homework_04_est,...,final_exam_attempt_2_eng,final_exam_attempt_2_est,final_exam_attempt_3_eng,final_exam_attempt_3_est,bonus,course_numeric_grade,course_letter_grade,year,active_days,result
0,TCS_2019,S027cf9,NaN,88.0,NaN,104.0,NaN,66.0,NaN,76.0,...,NaN,NaN,NaN,NaN,NaN,72.10,C,2019,18.0,pass
1,TCS_2019,S032255,NaN,0.0,NaN,NaN,22.0,NaN,NaN,NaN,...,102.0,NaN,NaN,NaN,NaN,67.70,C,2019,6.0,pass
2,TCS_2019,S03a8b8,NaN,99.0,NaN,108.0,NaN,110.0,NaN,105.0,...,NaN,NaN,NaN,NaN,NaN,107.02,A,2019,30.0,pass
3,TCS_2019,S04cf7f,NaN,88.0,NaN,110.0,NaN,88.0,NaN,76.0,...,NaN,NaN,NaN,NaN,NaN,85.85,A,2019,21.0,pass
4,TCS_2019,S05f285,NaN,88.0,NaN,86.0,NaN,66.0,NaN,108.0,...,NaN,NaN,NaN,NaN,NaN,37.15,F,2019,32.0,fail


### Train and test split

To test whether the model generalizes to newer academic years, the split is based on years rather than a random split.

- Training data: 2019 and 2021–2024
- Test data: 2025–2026
- 2020 is excluded according to the project plan.

In [6]:
train = data[data["year"].isin([2019, 2021, 2022, 2023, 2024])].copy()
test = data[data["year"].isin([2025, 2026])].copy()

print("Train:", train.shape)
print("Test:", test.shape)

print("\nTrain years:")
print(train["year"].value_counts().sort_index())

print("\nTest years:")
print(test["year"].value_counts().sort_index())

Train: (564, 44)
Test: (279, 44)

Train years:
year
2019    114
2021    142
2022     83
2023    106
2024    119
Name: count, dtype: int64

Test years:
year
2025    137
2026    142
Name: count, dtype: int64


In [7]:
print("Train results:")
print(train["result"].value_counts())

print("\nTest results:")
print(test["result"].value_counts())

Train results:
result
pass    482
fail     82
Name: count, dtype: int64

Test results:
result
pass    234
fail     45
Name: count, dtype: int64


## 2. Baseline model

As a simple baseline, we use the rule:

**If a student was active on 15 or fewer days, classify the student as being at risk of failing.**

The threshold is selected using only the training data. The test data is then used once to evaluate the fixed rule.

In [9]:
from sklearn.metrics import precision_score, recall_score

results = []

for threshold in range(1, int(train["active_days"].max()) + 1):
    y_true = (train["result"] == "fail").astype(int)
    y_pred = (train["active_days"] <= threshold).astype(int)

    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)

    results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision
    })

baseline_results = pd.DataFrame(results)

display(
    baseline_results[
        (baseline_results["recall"] >= 0.70) &
        (baseline_results["precision"] >= 1/3)
    ]
)

,threshold,recall,precision
15,16,0.707317,0.353659
16,17,0.719512,0.335227


In [10]:
threshold = 15

y_test = (test["result"] == "fail").astype(int)
y_pred = (test["active_days"] <= threshold).astype(int)

test_recall = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)

print("Baseline threshold:", threshold)
print("Test recall:", test_recall)
print("Test precision:", test_precision)

Baseline threshold: 15
Test recall: 0.7333333333333333
Test precision: 0.358695652173913


### Baseline result

Using `active_days <= 15` as a risk rule, the model achieved a recall of 0.733 and a precision of 0.359 on the 2025–2026 test data.

This means that the baseline identified about 73% of students who failed, and about 36% of students flagged as at risk actually failed.

## 3. Logistic Regression

As the first machine learning model, Logistic Regression was trained using `active_days` as the only feature.

The model predicts the probability that a student will fail the course.

In [13]:
#Logistic regression

from sklearn.linear_model import LogisticRegression

X_train = train[["active_days"]]
y_train = (train["result"] == "fail").astype(int)

X_test = test[["active_days"]]
y_test = (test["result"] == "fail").astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

train_prob = model.predict_proba(X_train)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(test_prob[:10])

First 10 predicted probabilities:
[0.07125409 0.10343673 0.01689848 0.25508423 0.63650337 0.33990825
 0.04263339 0.16580663 0.20690878 0.14784116]


In [14]:
from sklearn.metrics import roc_auc_score

train_auc = roc_auc_score(y_train, train_prob)
test_auc = roc_auc_score(y_test, test_prob)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)

Train AUC: 0.7974521809533448
Test AUC: 0.8131054131054131


### Logistic Regression AUC

The Logistic Regression model achieved a test AUC of 0.813.

This indicates that `active_days` contains useful information for distinguishing students who pass from students who fail.

### Selecting the classification threshold

Logistic Regression produces a probability of failure for each student. To turn these probabilities into risk labels, we need to choose a classification threshold.

We select the threshold using only the training data. We look for thresholds that satisfy the project criteria:

- recall ≥ 0.70
- precision ≥ 1/3

The test data is not used when choosing the threshold.

In [ ]:
threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):
    y_pred = (train_prob >= threshold).astype(int)

    recall = recall_score(y_train, y_pred, zero_division=0)
    precision = precision_score(y_train, y_pred, zero_division=0)

    threshold_results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision
    })

logreg_thresholds = pd.DataFrame(threshold_results)

display(
    logreg_thresholds[
        (logreg_thresholds["recall"] >= 0.70) &
        (logreg_thresholds["precision"] >= 1/3)
    ]
)

### Threshold selection result

Several thresholds satisfy both project criteria on the training data. We select a threshold of 0.19 for the final evaluation on the 2025–2026 test data.

In [16]:
threshold = 0.19

y_test_pred = (test_prob >= threshold).astype(int)

test_recall = recall_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)

print("Logistic Regression threshold:", threshold)
print("Test recall:", test_recall)
print("Test precision:", test_precision)

Logistic Regression threshold: 0.19
Test recall: 0.7555555555555555
Test precision: 0.3300970873786408


### Logistic Regression result

Logistic Regression using only `active_days` achieved a test AUC of 0.813. Using a probability threshold of 0.19, the model achieved a recall of 0.756 and a precision of 0.330 on the 2025–2026 test data.

The model improved recall compared with the baseline, but its precision was slightly below the required 1/3 criterion.

## 4. Preliminary model comparison

The first two approaches are compared on the 2025–2026 test data.

In [23]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression"
    ],
    "Test AUC": [
        np.nan,
        test_auc
    ],
    "Test Recall": [
        0.7333333333,
        test_recall
    ],
    "Test Precision": [
        0.3586956522,
        test_precision
    ]
})

display(comparison)

,Model,Test AUC,Test Recall,Test Precision
0,Baseline,NaN,0.733333,0.358696
1,Logistic Regression,0.813105,0.755556,0.330097
